### Station Grouping: K=5 Nearest Neighbors

In [ ]:
import pandas as pd
import numpy as np
import os

In [ ]:
# Load all station CSVs and concatenate with year column
frames = []
for year in range(2014, 2018):
    path = f'../Data/Stations_{year}.csv'
    df = pd.read_csv(path)
    df['year'] = year
    frames.append(df)
    print(f'{year}: {len(df)} stations loaded')

stations_all = pd.concat(frames, ignore_index=True)
print(f'\nTotal rows: {len(stations_all)}')
print(f'Unique station names: {stations_all["name"].nunique()}')
stations_all.head()

In [ ]:
# Deduplicate by name, compute mean lat/long per station
station_coords = stations_all.groupby('name').agg(
    latitude=('latitude', 'mean'),
    longitude=('longitude', 'mean')
).reset_index()

print(f'Unique stations with coordinates: {len(station_coords)}')
station_coords.head()

In [ ]:
def haversine(lat1, lon1, lat2, lon2):
    """
    Calculate the Haversine distance between two points
    on Earth given their latitude and longitude in degrees.
    Returns distance in meters.
    """
    R = 6371000  # Earth's radius in meters

    lat1_rad = np.radians(lat1)
    lat2_rad = np.radians(lat2)
    dlat = np.radians(lat2 - lat1)
    dlon = np.radians(lon2 - lon1)

    a = np.sin(dlat / 2) ** 2 + \
        np.cos(lat1_rad) * np.cos(lat2_rad) * np.sin(dlon / 2) ** 2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))

    return R * c

In [ ]:
# Define target station names
target_mont_royal = "M\u00e9tro Mont-Royal (Rivard / du Mont-Royal)"
target_berri = "Berri / de Maisonneuve"

# Get coordinates for target stations
mont_royal_row = station_coords[station_coords['name'] == target_mont_royal].iloc[0]
berri_row = station_coords[station_coords['name'] == target_berri].iloc[0]

mont_royal_lat, mont_royal_lon = mont_royal_row['latitude'], mont_royal_row['longitude']
berri_lat, berri_lon = berri_row['latitude'], berri_row['longitude']

print(f'Mont-Royal: lat={mont_royal_lat}, lon={mont_royal_lon}')
print(f'Berri: lat={berri_lat}, lon={berri_lon}')

In [ ]:
# Compute distances from every station to Mont-Royal and to Berri
station_coords['dist_to_mont_royal'] = station_coords.apply(
    lambda row: haversine(row['latitude'], row['longitude'], mont_royal_lat, mont_royal_lon),
    axis=1
)

station_coords['dist_to_berri'] = station_coords.apply(
    lambda row: haversine(row['latitude'], row['longitude'], berri_lat, berri_lon),
    axis=1
)

station_coords[['name', 'dist_to_mont_royal', 'dist_to_berri']].head(10)

In [ ]:
K = 5

# K nearest neighbors for Mont-Royal (excluding the target itself)
mont_royal_neighbors = station_coords[
    station_coords['name'] != target_mont_royal
].nsmallest(K, 'dist_to_mont_royal')[['name', 'dist_to_mont_royal']]

# K nearest neighbors for Berri (excluding the target itself)
berri_neighbors = station_coords[
    station_coords['name'] != target_berri
].nsmallest(K, 'dist_to_berri')[['name', 'dist_to_berri']]

print(f'Mont-Royal: {K} nearest neighbors')
print(mont_royal_neighbors.to_string(index=False))
print(f'\nBerri: {K} nearest neighbors')
print(berri_neighbors.to_string(index=False))

In [ ]:
# Print the station groups nicely
print('Mont-Royal group (6 stations):')
print(f'  1. {target_mont_royal} [target] - 0m')
for i, (_, row) in enumerate(mont_royal_neighbors.iterrows(), start=2):
    print(f'  {i}. {row["name"]} - {row["dist_to_mont_royal"]:.0f}m')

print(f'\nBerri group (6 stations):')
print(f'  1. {target_berri} [target] - 0m')
for i, (_, row) in enumerate(berri_neighbors.iterrows(), start=2):
    print(f'  {i}. {row["name"]} - {row["dist_to_berri"]:.0f}m')

In [ ]:
# Build station_groups DataFrame
groups_rows = []

# Mont-Royal group: target + 5 neighbors
groups_rows.append({
    'group': 'mont_royal',
    'station_name': target_mont_royal,
    'is_target': True,
    'distance_m': 0.0
})
for _, row in mont_royal_neighbors.iterrows():
    groups_rows.append({
        'group': 'mont_royal',
        'station_name': row['name'],
        'is_target': False,
        'distance_m': row['dist_to_mont_royal']
    })

# Berri group: target + 5 neighbors
groups_rows.append({
    'group': 'berri',
    'station_name': target_berri,
    'is_target': True,
    'distance_m': 0.0
})
for _, row in berri_neighbors.iterrows():
    groups_rows.append({
        'group': 'berri',
        'station_name': row['name'],
        'is_target': False,
        'distance_m': row['dist_to_berri']
    })

station_groups = pd.DataFrame(groups_rows)
print(f'station_groups shape: {station_groups.shape}')
station_groups

In [ ]:
# Build year-to-code lookup
# For each group station, find its code in each year
lookup_rows = []
for _, row in station_groups.iterrows():
    name = row['station_name']
    group = row['group']
    for year in range(2014, 2018):
        year_stations = stations_all[
            (stations_all['name'] == name) & (stations_all['year'] == year)
        ]
        for _, s in year_stations.iterrows():
            lookup_rows.append({
                'group': group,
                'station_name': name,
                'year': year,
                'station_code': s['code']
            })

station_code_lookup = pd.DataFrame(lookup_rows)
print(f'station_code_lookup shape: {station_code_lookup.shape}')

In [ ]:
# Display the lookup table and verify counts
print('Rows per group:')
print(station_code_lookup.groupby('group').size())
print(f'\nExpected: ~6 stations x 4 years = ~24 rows per group')
print(f'(fewer if some stations don\'t exist in all years)\n')
station_code_lookup

In [ ]:
# Save outputs
os.makedirs('../Project_datasets', exist_ok=True)

station_groups.to_csv('../Project_datasets/station_groups.csv', index=False)
station_code_lookup.to_csv('../Project_datasets/station_code_lookup.csv', index=False)
print('Saved station_groups.csv and station_code_lookup.csv')

In [ ]:
# Verification - print summary stats
print('=== Summary ===')
print(f'Total unique stations across all years: {stations_all["name"].nunique()}')
print(f'Stations in Mont-Royal group: {len(station_groups[station_groups["group"] == "mont_royal"])}')
print(f'Stations in Berri group: {len(station_groups[station_groups["group"] == "berri"])}')
print(f'\nLookup table entries: {len(station_code_lookup)}')
print(f'  Mont-Royal group: {len(station_code_lookup[station_code_lookup["group"] == "mont_royal"])} (station-year combos)')
print(f'  Berri group: {len(station_code_lookup[station_code_lookup["group"] == "berri"])} (station-year combos)')
print(f'\nYears covered per group station:')
print(station_code_lookup.groupby(['group', 'station_name'])['year'].apply(list).to_string())
print(f'\nOutput files:')
print(f'  ../Project_datasets/station_groups.csv ({os.path.getsize("../Project_datasets/station_groups.csv")} bytes)')
print(f'  ../Project_datasets/station_code_lookup.csv ({os.path.getsize("../Project_datasets/station_code_lookup.csv")} bytes)')